In [1]:
import pandas as pd
import geopandas as gpd
import json
from shapely.affinity import translate
from helpers import remove_points_and_lines, calculate_intersection

In [2]:
def clean_ba_zone_geo(ba_system_geo, ba_zone_geo):
    unclaimed_areas = (
        ba_system_geo.geometry
        .difference(ba_zone_geo.union_all())
        .reset_index()
    )
    
    unclaimed_county_geo = calculate_intersection(
        unclaimed_areas,
        county_geo,
        ['rb']
    )
    
    unclaimed_counties = list(unclaimed_county_geo.rb.unique())
    
    zone_county_intersects = calculate_intersection(
        ba_zone_geo,
        county_geo.loc[county_geo.rb.isin(unclaimed_counties)],
        ['zone', 'rb']
    )
    zone_county_intersects['area'] = zone_county_intersects.geometry.area
    zone_county_intersects = (
        zone_county_intersects.sort_values('area', ascending=False)
        .drop_duplicates(subset='rb', keep='first')
    )
    county_zone_map = dict(zip(zone_county_intersects['rb'], zone_county_intersects['zone']))
    
    ba_county_geo_unassigned = county_geo.loc[(
        county_geo.rb.isin(unclaimed_counties) & ~county_geo.rb.isin(county_zone_map.keys())
    )]
    zone_county_nearest = gpd.sjoin_nearest(ba_zone_geo, ba_county_geo_unassigned, how='right')
    county_zone_map = (
        county_zone_map | dict(zip(zone_county_nearest['rb'], zone_county_nearest['zone']))
    )
    
    # Assign all unclaimed areas to their applicable zones
    unclaimed_county_geo['zone'] = unclaimed_county_geo['rb'].map(county_zone_map)
    unclaimed_area_geo = remove_points_and_lines(
        unclaimed_county_geo.dissolve('zone').reset_index(), ['zone']
    )
    
    # Append previously unclaimed areas to zone geo
    ba_zone_geo_all = remove_points_and_lines(
        pd.concat([ba_zone_geo, unclaimed_area_geo])
        .dissolve('zone')
        .reset_index(),
        ['zone']
    )
    ba_zone_geo_all = calculate_intersection(
        ba_zone_geo_all[['zone', 'geometry']],
        ba_system_geo,
        ['zone']
    )

    return ba_zone_geo_all

In [ ]:
county_geo = gpd.read_file('../data/shapefiles/US_COUNTY_2022')
bas = gpd.read_file('../data/shapefiles/Balancing_Authorities').to_crs(county_geo.crs)
erst = gpd.read_file("../data/shapefiles/Electric_Retail_Service_Territories").to_crs(county_geo.crs)

with open('config/rto_region_eia_code_map.json', 'r') as file:
    rto_region_id_eia_code_map = json.load(file)

rto_regions = gpd.read_file('../data/shapefiles/RTO_Regions').to_crs(county_geo.crs)
rto_regions['EIAcode'] = rto_regions['Unique_ID'].map(rto_region_id_eia_code_map)
subbas = rto_regions.dropna(subset='EIAcode')
miso_new = subbas.loc[subbas.RTO_ISO == 'MISO'].dissolve(['EIAcode']).reset_index()
subbas = pd.concat([subbas.loc[subbas.RTO_ISO != 'MISO'], miso_new], ignore_index=True)

eia_930_ref = pd.read_excel('../data/EIA930_Reference_Tables.xlsx', sheet_name='BA Subregions')
subba_code_name_map = dict(zip(eia_930_ref['BA Subregion Code'], eia_930_ref['BA Subregion Name']))

In [4]:
pnm_new = gpd.overlay(
    bas.loc[bas.EIAcode == 'PNM'],
    county_geo.loc[county_geo.STATE == 'Oregon'].dissolve(),
    how='difference'
)
bas = pd.concat([bas.loc[bas.EIAcode != 'PNM'], pnm_new], ignore_index=True)

In [5]:
walc_new = calculate_intersection(
    bas.loc[bas.EIAcode == 'WALC'],
    county_geo.loc[(
        county_geo.STATE.isin(['Arizona', 'New Mexico', 'Utah'])
        | county_geo.rb.isin(['p06025', 'p06065', 'p06071', 'p32003'])
    )],
    None
)
bas = pd.concat([bas.loc[bas.EIAcode != 'WALC'], walc_new], ignore_index=True)

In [ ]:
spp_region = rto_regions.loc[((rto_regions.RTO_ISO == 'SPP') & (rto_regions.Map_Type == 'Region'))].head(1)

spp_fill_in_features = gpd.read_file('../data/shapefiles/spp_manually_drawn_zones')
spp_fill_in_features['geometry'] = (
    spp_fill_in_features['geometry']
    .apply(lambda geom: translate(geom, yoff=20000))
    .make_valid()
)
id_name_map = {
    2: 'WAUE',
    3: 'WAUE',
    99: 'SPS',
    98: 'CSWS',
    97: 'WFEC',
    95: 'OKGE',
    94: 'OKGE',
    93: 'CSWS',
    92: 'CSWS',
    91: 'CSWS',
    90: 'CSWS',
    89: 'GRDA',
    88: 'CSWS',
    87: 'EDE',
    85: 'MPS',
    81: 'SECI',
}
spp_fill_in_features['EIAcode'] = spp_fill_in_features['id'].map(id_name_map)
spp_fill_in_features = spp_fill_in_features.dissolve('EIAcode')

zones_no_overlap = {}
for zone, geo in spp_fill_in_features['geometry'].items():
    zones_no_overlap[zone] = geo.difference(spp_fill_in_features.drop(zone).union_all())

spp_fill_in_features = gpd.GeoDataFrame({
    'EIAcode': zones_no_overlap.keys(),
    'geometry': zones_no_overlap.values()
}, crs=spp_fill_in_features.crs)

# WAUE
waue_non_ne = calculate_intersection(
    spp_region,
    county_geo.loc[county_geo.STCODE.isin(['MT', 'ND', 'SD', 'MN', 'IA', 'WY'])].dissolve(),
    []
)
waue_ne = spp_fill_in_features.loc[spp_fill_in_features.EIAcode == 'WAUE'].copy()
waue_new = (
    pd.concat([waue_non_ne, waue_ne])
    .dissolve()
)
subbas.loc[subbas.EIAcode == 'WAUE', 'geometry'] = waue_new.iloc[0]['geometry']

# NPPD
ne_counties = county_geo.loc[county_geo.STATE == 'Nebraska'].copy()
ne_counties['county_area'] = ne_counties['geometry'].area
nppd_new = gpd.overlay(
    ne_counties,
    subbas.loc[subbas.EIAcode.isin(['WAUE', 'OPPD', 'LES'])].dissolve(),
    how='difference'
)
nppd_new['proportion_of_county_area'] = nppd_new.geometry.area / nppd_new['county_area']
nppd_new = (
    nppd_new.loc[nppd_new.proportion_of_county_area.round(1) > 0]
    .dissolve()
)
subbas.loc[subbas.EIAcode == 'NPPD', 'geometry'] = nppd_new.iloc[0]['geometry']

# Drawn features
spp_fill_in_features_sub = (
    spp_fill_in_features.loc[~spp_fill_in_features.EIAcode.isin(['WAUE', 'SECI', 'MPS'])]
)
for eia_code in spp_fill_in_features_sub.EIAcode.tolist():
    subbas.loc[subbas.EIAcode == eia_code, 'geometry'] = (
        spp_fill_in_features_sub.loc[spp_fill_in_features_sub.EIAcode == eia_code].iloc[0]['geometry']
    )

# WR
ks_counties = county_geo.loc[county_geo.STATE == 'Kansas'].copy()
ks_counties['county_area'] = ks_counties['geometry'].area
wr_new = gpd.overlay(
    ks_counties,
    subbas.loc[(subbas.RTO_ISO == 'SPP') & (subbas.EIAcode != 'WR')].dissolve(),
    how='difference'
)
ks_counties = county_geo.loc[county_geo.STATE == 'Kansas'].copy()
ks_counties['county_area'] = ks_counties['geometry'].area
wr_new = gpd.overlay(
    ks_counties,
    subbas.loc[(subbas.RTO_ISO == 'SPP') & (subbas.EIAcode != 'WR')].dissolve(),
    how='difference'
)
wr_new['proportion_of_county_area'] = wr_new.geometry.area / wr_new['county_area']
wr_new = (
    wr_new.loc[wr_new.proportion_of_county_area.round(1) > 0]
    .dissolve()
)
subbas.loc[subbas.EIAcode == 'WR', 'geometry'] = wr_new.iloc[0]['geometry']

# EDE
subbas.loc[subbas.EIAcode == 'EDE', 'geometry'] = (
    gpd.overlay(
        subbas.loc[subbas.EIAcode == 'EDE'],
        subbas.loc[(subbas.RTO_ISO == 'SPP') & (subbas.EIAcode != 'EDE')].dissolve(),
        how='difference'   
    )
    .iloc[0]
    ['geometry']
)

# Combine everything
ba_system_geo = spp_region.copy()
ba_zone_geo = (
    subbas.loc[subbas.RTO_ISO == 'SPP']
    .copy()
    .rename(columns={'EIAcode': 'zone'})
)
spp_zones = clean_ba_zone_geo(ba_system_geo, ba_zone_geo)
subbas = (
    pd.concat([
        subbas.loc[subbas.RTO_ISO != 'SPP'],
        spp_zones.drop(columns='EIAcode').rename(columns={'zone': 'EIAcode'})
    ], ignore_index=True)
)

In [7]:
ba_system_geo = bas.loc[bas.EIAcode == 'PJM'].copy()
pjm_old = subbas.loc[subbas.RTO_ISO == 'PJM']
pjm_zones = pjm_old.copy()

# AP
ap_old = pjm_zones.loc[pjm_zones.LOC_ABBREV == 'APS'].copy()
ap_new = gpd.overlay(
    ap_old,
    county_geo.loc[county_geo.STCODE == 'VA'].dissolve(),
    how='difference'
)

va_geo = (
    county_geo.loc[county_geo.STCODE == 'VA']
    .dissolve()
    ['geometry']
    .iloc[0]
)
va_no_dom_geo = gpd.GeoDataFrame(
    geometry=[va_geo.difference(erst.loc[erst.ID == '19876'].iloc[0]['geometry'])],
    crs=erst.crs
)
va_counties_no_dom_geo = calculate_intersection(
    va_no_dom_geo,
    county_geo.loc[county_geo.STCODE == 'VA'],
    ['rb']
)
pe_transmission_zone_geo = (
    va_counties_no_dom_geo.loc[(
        va_counties_no_dom_geo.NAME.isin([
            'Frederick',
            'Winchester',
            'Clarke',
            'Warren',
            'Page',
            'Rappahannock',
            'Madison',
            'Highland',
            'Greene'
        ])
    )]
    .dissolve()
)
ap_new = pd.concat([ap_new, pe_transmission_zone_geo]).dissolve()
pjm_zones.loc[pjm_zones.LOC_ABBREV == 'APS', 'geometry'] = ap_new.iloc[0]['geometry']

# AEP
aep_old = pjm_zones.loc[pjm_zones.LOC_ABBREV == 'AEP'].copy()
aep_new = (
    pd.concat([
        aep_old,
        county_geo.loc[county_geo.rb.isin(['p51195', 'p51720', 'p26159', 'p26077', 'p26149', 'p26027', 'p26021'])]
    ])
    .dissolve()
)
pjm_zones.loc[pjm_zones.LOC_ABBREV == 'AEP', 'geometry'] = aep_new.iloc[0]['geometry']

# DPL
dpl_old = pjm_zones.loc[pjm_zones.LOC_ABBREV == 'DPL']
dpl_new = pd.concat([
    dpl_old,
    county_geo.loc[county_geo.rb.isin(['p51001', 'p51131'])]
]).dissolve()
pjm_zones.loc[pjm_zones.LOC_ABBREV == 'DPL', 'geometry'] = dpl_new.iloc[0]['geometry']

# DOM
dom_old = pjm_zones.loc[pjm_zones.LOC_ABBREV == 'DOM']
dom_nc = calculate_intersection(
    dom_old,
    county_geo.loc[county_geo.STCODE == 'NC'].dissolve(),
    None
)
dom_va = gpd.overlay(
    county_geo.loc[county_geo.STCODE == 'VA'].dissolve(),
    pjm_zones.loc[pjm_zones.LOC_ABBREV != 'DOM'].dissolve(),
    how='difference'
)
dom_new = pd.concat([dom_nc, dom_va]).dissolve()
pjm_zones.loc[pjm_zones.LOC_ABBREV == 'DOM', 'geometry'] = dom_new.iloc[0]['geometry']

# PEP
pep_geo = (
    pd.concat([
        gpd.overlay(
            county_geo.loc[(county_geo.rb.isin(['p24037', 'p24017', 'p24009', 'p24033', 'p24031', 'p24027']))].dissolve(),
            pjm_zones.dissolve(),
            how='difference'
        ),
        erst.loc[erst.STATE == 'DC']
    ])
    .dissolve()
    [['geometry']]
)
pjm_zones.loc[pjm_zones.LOC_ABBREV == 'PEPCO', 'geometry'] = pep_geo.iloc[0]['geometry']

# ATSI
atsi_old = pjm_zones.loc[pjm_zones.LOC_ABBREV == 'ATSI']
atsi_add = gpd.overlay(
    county_geo.loc[county_geo.rb.isin(['p42039', 'p42085', 'p42073', 'p42007', 'p42019', 'p42003'])].dissolve(),
    pjm_zones.dissolve(),
    how='difference'
)
atsi_new = pd.concat([atsi_old, atsi_add]).dissolve()
pjm_zones.loc[pjm_zones.LOC_ABBREV == 'ATSI', 'geometry'] = atsi_new.iloc[0]['geometry']

# PPL
ppl_old = pjm_zones.loc[pjm_zones.LOC_ABBREV == 'PPL']
ppl_add = gpd.overlay(
    county_geo.loc[county_geo.rb.isin(['p42079', 'p42131'])].dissolve(),
    pjm_zones.dissolve(),
    how='difference'
)
ppl_new = pd.concat([ppl_old, ppl_add]).dissolve()
pjm_zones.loc[pjm_zones.LOC_ABBREV == 'PPL', 'geometry'] = ppl_new.iloc[0]['geometry']

# AC
ac_new = (
    pd.concat([
        pjm_zones.loc[pjm_zones.LOC_ABBREV == 'AECO'],
        county_geo.loc[county_geo.rb.isin(['p34011'])]
    ]).dissolve()
)
pjm_zones.loc[pjm_zones.LOC_ABBREV == 'AECO', 'geometry'] = ac_new.iloc[0]['geometry']

# Combine everything
pjm_zones = calculate_intersection(
    ba_system_geo,
    pjm_zones,
    'LOC_ABBREV'
)
subbas = (
    pd.concat([
        subbas.loc[subbas.RTO_ISO != 'PJM'],
        pjm_zones.rename(columns={'EIAcode_2': 'EIAcode'})
    ], ignore_index=True)
)

In [ ]:
subbas['EIAname'] = subbas['EIAcode'].map(subba_code_name_map)

eia_930_ref = pd.read_excel('../data/EIA930_Reference_Tables.xlsx', sheet_name='BAs')
inactive_bas = eia_930_ref.loc[eia_930_ref['Active BA'] == 'No']['BA Code'].tolist()
inactive_bas.remove('WAUE')
gen_only_bas = eia_930_ref.loc[eia_930_ref['Generation Only BA'] == 'Yes']['BA Code'].tolist()
non_us_bas = eia_930_ref.loc[eia_930_ref['U.S. BA'] == 'No']['BA Code'].tolist()
parent_bas = ['CISO', 'ERCO', 'ISNE', 'MISO', 'NYIS', 'PJM', 'SWPP']
remove_bas = list(set(inactive_bas + gen_only_bas + non_us_bas + parent_bas + ['OVEC', 'SEC']))

eia_930_regions = (
    pd.concat([bas.loc[~bas.EIAcode.isin(remove_bas)], subbas], ignore_index=True)
    [['EIAcode', 'EIAname', 'geometry']]
)

In [9]:
eia_930_regions['geometry'] = eia_930_regions.make_valid()

In [ ]:
eia_930_regions.to_file('../data/shapefiles/bas_and_subbas')